Example code and instructions courtesy of The AI Guy.

Watch his video tutorial here:

https://www.youtube.com/watch?v=10joRJt39Ns

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# This creates a handy symbolic link to Google Drive

In [ ]:
!ln -s /content/gdrive/MyDrive/ /mydrive
!ls /mydrive

# Helper Functions

In [ ]:
def imShow(path):
  import cv2
  import matplotlib.pyplot as plt
  %matplotlib inline

  image = cv2.imread(path)
  height, width = image.shape[:2]
  resized_image = cv2.resize(image,(3*width, 3*height), interpolation = cv2.INTER_CUBIC)

  fig = plt.gcf()
  fig.set_size_inches(18, 10)
  plt.axis("off")
  plt.imshow(cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB))
  plt.show()

def download(path):
  from google.colab import files
  files.download(path)

# Unzip Darknet...
...if uploading a previously saved and and compressed copy.

_NOTE: If darknet unzips into the /content folder, drag it out into the top-level directory._

In [ ]:
# !unzip darknet.zip

# Otherwise...
The following cells will clone darknet from AlexeyAB's famous repository, adjust the Makefile to enable OPENCV and GPU for darknet and then build darknet.

Do not worry about any warnings when you run the '!make' cell!

In [ ]:
# clone darknet repo
!git clone https://github.com/AlexeyAB/darknet

In [ ]:
# change makefile to have GPU and OPENCV enabled
# comment out the last two lines if using CPU (not recommended for training)
%cd darknet
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile

In [ ]:
# verify CUDA
!/usr/local/cuda/bin/nvcc --version

In [ ]:
# make darknet (build)
!make

# Force an error at this cell if using "run all" to execute initial setup

In [ ]:
# This stops 'Run all' at this cell by causing an error
# assert False

# The following cells will zip and download a copy of the Darknet folder to disk

In [ ]:
# !zip -r /content/darknet.zip /content/darknet

In [ ]:
# from google.colab import files
# files.download("/content/darknet.zip")

# Training YOLOv3

In order to create a custom YOLOv3 detector you will need the following:

*   Labeled Custom Dataset
*   Custom .cfg file
*   obj.data and obj.names files
*   train.txt file (test.txt is optional here as well)

## Import and unzip `obj` folder from Google Drive

In [ ]:
#%cd darknet

In [ ]:
!cp /mydrive/yolov3/obj.zip ../

In [ ]:
!unzip ../obj.zip -d data/

## Custom Dataset and Annotations

The labeled custom dataset consists of image files and annotation text files. Each image file is paired with a corresponding text file that contains the annotations for the bounding boxes in the image. Each text file should have the same name as the image file it is paired with but with a .txt extension. Each line in the text files represents a single bounding box within the corresponding image. Bounding box annotations should be formatted as such:

`<class_id> <x> <y> <box_width> <box_height>`

Example text file containg two bounding boxes:

```
1 0.1578 0.5245 0.1510 0.2398
0 0.7382 0.5148 0.0661 0.2333
```

NOTE: x and y values should be the coordinates of the _center_ of a bounding box rather than its top left corner. Also, _x_ and _box_width_ should be normalized by dividing them by the image's total width, and _y_ and _box_height_ should be normalized by the image's total height.

Image and text files should be contained in a folder named `obj`.

## Custom .cfg File

First, make a copy of ```yolov3.cfg``` for customization.

In [ ]:
# make sure you are in the darknet folder
# %cd darknet

In [ ]:
# download yolov3.cfg to google drive and change its name
# !cp cfg/yolov3.cfg /mydrive/yolov3/yolov3_custom.cfg

In [ ]:
# to download to local machine (change its name to yolov3_custom.cfg once you download)
# download('cfg/yolov3.cfg')

## Use a text/code editor to edit `yolov3_custom.cfg`.

For training, comment lines 3 and 4 (the two lines under testing) and uncomment lines 6 and 7 (the two lines under training). For testing, uncomment the two testing lines and comment the two training lines.

Set `max_batches` (line 20) equal to 2000 * number_of_classes. E.g., for 4 classes, `max_batches=8000`.

Set `steps` (line 22) equal to 0.8(max_batches),0.9(max_batches). E.g., for 4 classes (max_batches=8000), `steps=6400,7200`.

On lines 610, 696, and 783, set `classes` equal to the number of classes in your dataset.

On lines 603, 689, and 776, set `filters` equal to (5 + number_of_classes) * 3. E.g., for 4 classes, `filters = (5 + 4) * 3 = 27`.

NOTE: Leaving `random=1` on lines 615, 701, and 788 will allow YOLO to apply random transformations to training images to bolster training variety and improve model performance at the cost of increased training time.

After creating and editing the `yolov3_custom.cfg` file, place it in the `darknet/cfg` folder or upload it to Google Drive and run the following cell to copy it over.

In [ ]:
# upload the custom .cfg back to cloud VM from Google Drive
!cp /mydrive/yolov3/yolov3_custom_train.cfg ./cfg
!cp /mydrive/yolov3/yolov3_custom_test.cfg ./cfg

## Create `obj.data` file

`obj.data` contains
- The number of classes in the dataset
- The location of the `train.txt`file
- The location of the `test.txt` file (optional)
- The location of the `obj.names` file
- The folder where YOLO will save the weights it learns after every 100 epochs, 1000 epochs, and after taining is complete.

Example `obj.data` file:
```
classes=4
train=data/train.txt
valid=data/test.txt
names=data/obj.names
backup=/mydrive/yolov3/backup
```

## Create `obj.names` file

The `obj.names` file lists the names of the classes in the dataset in the same order as their numeric ids.

Example `obj.names` file:
```
zss
cloud
ganon
younglink
```

After creating the `obj.data` and `obj.names` files, place them in the `darknet/data` folder or upload them to Google Drive and run the following cell to copy them over.

In [ ]:
# upload the obj.names and obj.data files to cloud VM from Google Drive
!cp /mydrive/yolov3/obj.names ./data
!cp /mydrive/yolov3/obj.data  ./data

## Create the `train.txt` file (and optionally `test.txt`)

The `train.txt` file contains a line-delimited list of paths to each training image in the `obj` folder. 

Example `train.txt` file:
```
data/obj/img1.jpg
data/obj/img2.jpg
data/obj/img3.jpg
data/obj/img4.jpg
...
data/obj/imgN.jpg
```

NOTE: This file should not include paths to annotation text files.

In [ ]:
pwd

In [ ]:
!cp /mydrive/yolov3/train.txt ./data

# Moving Your Custom Dataset Into Your Cloud VM
So now that you have your dataset properly formatted to be used for training we need to move it into this cloud VM so that when it comes the time we can actually use it for training.

I recommend renaming the folder with your images and text files on your local machine to be called '**obj**' and then creating a .zip folder of the 'obj' folder. Then I recommend uploading the zip to your Google Drive. So you should now have obj.zip someplace in your Google drive. 

This will **greatly reduce** the time it takes to transfer our dataset into our cloud VM.

Now we can copy in the zip and unzip it on your cloud VM.

In [ ]:
# this is where my zip is stored (I created a yolov3 folder where I will get my required files from)
# !ls /mydrive/yolov3

In [ ]:
# copy the .zip file into the root directory of cloud VM
# !cp /mydrive/yolov3/obj.zip ../

In [ ]:
# unzip the zip file and its contents should now be in /darknet/data/obj
# !unzip ../obj.zip -d data/

# Download pre-trained weights for the convolutional layers.
This gives the model an initial starting point and helps it converge faster.

In [ ]:
# upload pretrained convolutional layer weights
# !wget http://pjreddie.com/media/files/darknet53.conv.74

# Train Your Custom Object Detector
Run the cell below to begin training your YOLOv3 model. (The dont_show flag stops a chart from popping up since cloud can't open images on the spot)

**TIP:** This training could take several hours depending on how many iterations you chose in the .cfg file. You will want to let this run as you sleep or go to work for the day, etc. However, Colab Cloud Service kicks you off it's VMs if you are idle for too long (30-90 mins).

To avoid this hold (CTRL + SHIFT + i) at the same time to open up the inspector view on your browser.

Paste the following code into your console window and hit **Enter**. This function will click the screen every 10 minutes so that you don't get kicked off for being idle; however, Google has become wise to this and will occasionally throw the "I'm not a robot" dialog at you, so check back periodically!
```
function ClickConnect(){
console.log("Working"); 
document.querySelector("colab-toolbar-button#connect").click() 
}
setInterval(ClickConnect,60000)
```

In [ ]:
# train your custom detector
# !./darknet detector train data/obj.data cfg/yolov3_custom.cfg darknet53.conv.74 -dont_show

You can observe a chart of how your model did throughout the training process by running the below command. It shows a chart of your average loss vs. iterations. For your model to be 'accurate' you would aim for a loss under 2.

In [ ]:
# imShow('chart.png')

**TRICK**: If for some reason you get an error or your Colab goes idle during training, you have not lost your weights! Every 100 iterations a weights file called **yolov3_custom_last.weights** is saved to **mydrive/yolov3/backup/** folder (wherever your backup folder is). This is why we created this folder in our Google drive and not on the cloud VM. If your runtime crashes and your backup folder was in your cloud VM you would lose your weights and your training progress.

To kick off training from the last saved weights file, run the following cell but with your backup location.

In [ ]:
!./darknet detector train data/obj.data yolov3_custom_train.cfg ../backup/yolov3_custom_train_last.weights -dont_show

# Run Your Custom Object Detector

First, set the custom cfg file to test mode.

In [ ]:
# !cp /mydrive/yolov3/yolov3_custom.cfg /content/darknet/cfg/

In [ ]:
# %cd cfg
# !sed -i 's/batch=64/batch=1/' yolov3_custom.cfg
# !sed -i 's/subdivisions=16/subdivisions=1/' yolov3_custom.cfg
# %cd ..

Then, run your custom detector with this command (upload an image to your google drive to test. The `-thresh` flag sets the confidence percentage the model must have before it displays a bounding box.)

In [ ]:
import os
test_images = os.listdir("/mydrive/yolov3/younglink,ganondorf,fspec")

for i in range(len(test_images)):
  t = test_images[i]
#  !./darknet detector test data/obj.data /mydrive/yolov3/yolov3_custom_test.cfg /mydrive/yolov3/backup/yolov3_custom_train_last.weights /mydrive/yolov3/younglink,ganondorf,fspec/$t -thresh 0.3 -dont_show
  !./darknet detector test data/obj.data /mydrive/yolov3/yolov3_move_custom.cfg /mydrive/yolov3/char_move_2500.weights -thresh 0.3 -dont_show -ext_output /mydrive/yolov3/younglink,ganondorf,fspec/$t
  !cp ./predictions.jpg ../gdrive/MyDrive/yolov3/predictions/{i}.jpg

In [ ]:
!./darknet detector test data/obj.data /mydrive/yolov3/yolov3_custom_test.cfg /mydrive/yolov3/char_move_2500.weights -thresh 0.3 -dont_show -ext_output /mydrive/yolov3/younglink,ganondorf,fspec/younglink,ganondorf,5715.jpg > output.txt

In [ ]:
f = open('output.txt', 'r')
output = f.readlines()


In [ ]:
import re

if 'left_x' not in output[-1] or 'left_x' not in output[-2]:
  o1 = re.sub(r'[\(:\)_%]', '', output[-1].replace('  ', '').replace('\t', ''))
  split = o1.split(' ')
  print(split[0] + ',' + split[2] + ',' + re.sub(r'[a-zA-Z]+', '', split[3]) + ',' + re.sub(r'[a-zA-Z]+', '', split[4]) + ',' + re.sub(r'[a-zA-Z\\n]+', '', split[5]))

In [ ]:
# !./darknet detector demo data/obj.data /mydrive/yolov3/yolov3_custom_test.cfg /mydrive/yolov3/backup/yolov3_custom_train_last.weights  /mydrive/yolov3/videos/younglink,ganondorf,bair.mp4 -thresh 0.3 -dont_show
# !cp ./predictions.jpg ../gdrive/MyDrive/yolov3/predictions/{i}.jpg